# Knowledge Graph from Dubai data

## Setup the file paths for the files for cell 2 and 14, after that Run All will create the graph in neo4j, Remember to setup neo4j details in env

### Imports

In [1]:
import os
from dotenv import load_dotenv
import json
import anthropic
import pandas as pd
import geopandas as gpd
from neo4j import GraphDatabase

### Load Data 

In [2]:
# Event Record Data 
emr = pd.read_csv("C:/Users/danie/Documents/internship/KG-road/road_map_data/emr_df_final.csv")

# Event Plan Data frame
emp = pd.read_csv("C:/Users/danie/Documents/internship/KG-road/road_map_data/emp_df_final.csv")

# Event Plan Command Data frame
empc = pd.read_csv("C:/Users/danie/Documents/internship/KG-road/road_map_data/empc_df_final.csv")

# VMS Data frame
vms = pd.read_csv("C:/Users/danie/Documents/internship/KG-road/road_map_data/vms_df_final.csv")


In [3]:
emr_links = emr[['up_link_id', 'dn_link_id']].copy()
emr_links['same_value'] = emr_links['up_link_id'] == emr_links['dn_link_id']
print(emr_links)

       up_link_id      dn_link_id  same_value
0  17840001971739  17840005917320       False
1  17840006097112  17840005948005       False
2  17840006090411  17840006097972       False
3  17840004265116  17840004719186       False
4  17840003591931  17840001783588       False
5  17840001587376  17840001424568       False


In [ ]:

emr['version'] = pd.to_numeric(emr['version'], errors='coerce')


emr = emr.sort_values(['id', 'version'], ascending=[True, False]).drop_duplicates(subset=['id'], keep='first')

In [5]:
print(empc["cmd_list"])

0      {"eqtNo":"E11DMSG07N","eqtType":"AA","msgTypeI...
1      {"eqtNo":"E11DMSG08N","eqtType":"AA","msgTypeI...
2      {"eqtNo":"D63DMSP02W","eqtType":"C","msgTypeId...
3      {"eqtNo":"E44DMSG02N","eqtType":"AA","msgTypeI...
4      {"eqtNo":"D63DMSP01E","eqtType":"C","msgTypeId...
                             ...                        
222    {"id":null,"version":null,"createdDate":"2024-...
223    {"id":null,"version":null,"createdDate":"2024-...
224    {"id":null,"version":null,"createdDate":"2024-...
225    {"id":null,"version":null,"createdDate":"2024-...
226    {"id":null,"version":null,"createdDate":"2024-...
Name: cmd_list, Length: 227, dtype: object


### Extract Response plan from json

In [ ]:
def extract_info(x):
    data = json.loads(x)
    return {
        "eqtNo": data.get("eqtNo"),
        "eqtType": data.get("eqtType"),
        "msgTypeId": data.get("msgTypeId"),
        "msgDesc1": data.get("msgDesc1"),
        "msgDesc2": data.get("msgDesc2"),
    }


info_df = empc["cmd_list"].apply(extract_info).apply(pd.Series)


empc = pd.concat([empc, info_df], axis=1)
empc = empc.drop(columns=["cmd_list"])

### Load in ENV

In [40]:
load_dotenv()

# NEO4J Driver
driver = GraphDatabase.driver(os.environ.get("NEO4J_URI"),
                                auth=(os.environ.get("NEO4J_USERNAME"),
                                os.environ.get("NEO4J_PASSWORD"))
)

# LLM Model
llm = anthropic.Anthropic(api_key = os.getenv("ANTHROPIC_API_KEY"))


### Structure of data

In [7]:
print(emr.columns)
print(emp.columns)
print(empc.columns)
print(vms.columns)

Index(['tran_date', 'event_no', 'tran_time', 'alarm_no', 'assignee',
       'auto_close', 'auto_update', 'cause', 'cause_oth', 'congt_end_time',
       'congt_start_time', 'created_by', 'created_date', 'det_time',
       'direction', 'direction_desc', 'dist_to_upnode', 'dn_link_id',
       'dn_node_desc', 'dn_node_id', 'dn_node_site_id', 'dn_point', 'end_time',
       'end_x_coor', 'end_y_coor', 'eqt_alarm_id', 'event_aware_time',
       'event_desc', 'event_flag', 'event_icon', 'event_over_time',
       'event_severity', 'event_state', 'event_sub_type_id', 'event_type_id',
       'external_event_id', 'id', 'idle_job_id', 'idle_trigger_id', 'is_congt',
       'is_obsolete', 'lane_blockage', 'lane_blockage_time', 'loc_code',
       'loc_desc', 'loc_ent_exit', 'loc_type', 'notify_resp_time',
       'on_scene_time', 'operation', 'plan_auto_activate', 'plan_delay_time',
       'plan_request_id', 'plan_status', 'q_end_link_id', 'q_end_point',
       'q_end_x_coor', 'q_end_y_coor', 'q_len_ch

## Event data

### Creation of json schema for Knowledge graph


Graph structure

![Graph Structure](GS2.0.png)

### Prompt

In [ ]:
# emr_columns = ", ".join(emr.columns)
# emp_columns = ", ".join(emp.columns)
# empc_columns = ", ".join(empc.columns)
# vms_columns = ", ".join(vms.columns)

# prompt = f"""Your only purpose is to create json schema to create knowledge graphs in NEO4J database using the columns from the provided data frames, ONLY return json schema, DO NOT explain anything, DO NOT add a title,
# Nodes I want are, event record, event plan, event plan command and VMS
# the properties of each node will be located in their respective data frames
# The relationships between these nodes will be, from event record[id] to event plan[event_id], from event plan[id] to event plan command[plan_id], from event plan command[eqt_site_id] to VMS[EQT_EXT_ID]

# Carefully inspect the columns of each dataframe to determine the properties for each node:
# - event record: use all columns from the 'emr' dataframe: {emr_columns}.
# - event plan: use all columns from the 'emp' dataframe: {emp_columns}.
# - event plan command: use all columns from the 'empc' dataframe: {empc_columns}.
# - VMS: use all columns from the 'vms' dataframe: {vms_columns}.

# For relationships:
# - event record[id] -> event plan[event_id] (type: HAS_PLAN)
# - event plan[id] -> event plan command[plan_id] (type: HAS_COMMAND)
# - event plan command[eqt_site_id] -> VMS[EQT_EXT_ID] (type: CONTROLS_VMS)

# Do not include any explanation, only return the JSON schema as specified:
# {{
#     "nodes": [
#         {{"label": "event record", "properties": [...]}},
#         {{"label": "event plan", "properties": [...]}},
#         {{"label": "event plan command", "properties": [...]}},
#         {{"label": "VMS", "properties": [...]}}

#     ],
#     "relationships": [
#         {{"type": "HAS_PLAN", "from": "event record", "to": "event plan", "properties": []}},
#         {{"type": "HAS_COMMAND", "from": "event plan", "to": "event plan command", "properties": []}},
#         {{"type": "CONTROLS_VMS", "from": "event plan command", "to": "VMS", "properties": []}}
#     ]
# }}
# Replace [...] with the actual column names from the respective dataframes.


# Return JSON format with:
# - nodes: [label, properties]
# - relationships: [type, from, to, properties]"""




### Generate schema from prompt

In [ ]:
# response = llm.messages.create(
#     model = "claude-3-5-haiku-latest",
#     max_tokens = 5000,
#     messages = [{"role" : "user", "content" : prompt}]
# )

# print(response.content[0].text)

### Generated schema

In [11]:
schema = {
    "nodes": [
        {
            "label": "event_record",
            "properties": ["tran_date", "event_no", "tran_time", "alarm_no", "assignee", "auto_close", "auto_update", "cause", "cause_oth", "congt_end_time", "congt_start_time", "created_by", "created_date", "det_time", "direction", "direction_desc", "dist_to_upnode", "dn_link_id", "dn_node_desc", "dn_node_id", "dn_node_site_id", "dn_point", "end_time", "end_x_coor", "end_y_coor", "eqt_alarm_id", "event_aware_time", "event_desc", "event_flag", "event_icon", "event_over_time", "event_severity", "event_state", "event_sub_type_id", "event_type_id", "external_event_id", "id", "idle_job_id", "idle_trigger_id", "is_congt", "is_obsolete", "lane_blockage", "lane_blockage_time", "loc_code", "loc_desc", "loc_ent_exit", "loc_type", "notify_resp_time", "on_scene_time", "operation", "plan_auto_activate", "plan_delay_time", "plan_request_id", "plan_status", "q_end_link_id", "q_end_point", "q_end_x_coor", "q_end_y_coor", "q_len_chng_rate", "q_length", "report_contact_name", "report_contact_number", "road_clear_time", "road_code", "road_name", "rsp_dispatch_status", "sensitivity_level", "source", "source_oth", "specific_data", "start_lsn", "start_time", "start_x_coor", "start_y_coor", "temp_event_no", "tunnel_x", "tunnel_y", "up_link_id", "up_node_desc", "up_node_id", "up_node_site_id", "up_point", "updated_by", "updated_date", "version", "zone_id"]
        },
        {
            "label": "event_plan",
            "properties": ["tran_date", "id", "tran_time", "created_by", "created_date", "eva_result", "event_id", "exp_plan_id", "implemented_date", "is_implemented", "is_obsolete", "is_predef_plan", "is_revert_plan", "linked_event_id", "operation", "plan_active_time", "plan_request_id", "plan_status", "plan_type", "response_no", "start_lsn", "updated_by", "updated_date", "version"]
        },
        {
            "label": "event_plan_command",
            "properties": ["tran_date", "id", "tran_time", "ack", "cmd_id", "cmd_status", "created_by", "created_date", "eqt_ctrl_type", "eqt_site_id", "is_obsolete", "operation", "plan_id", "start_lsn", "sys_id", "updated_by", "updated_date", "version", "eqtNo", "eqtType", "msgTypeId", "msgDesc1", "msgDesc2"]
        },
        {
            "label": "VMS",
            "properties": ["ID", "EQT_NO", "EQT_EXT_ID", "EQT_TYPE", "ROAD_NAME", "ROAD_CAT", "LONGITUDE", "LATITUDE", "HEIGHT", "SITE_ID", "SEGMENT_ID", "LINK_ID", "DIST_TO_UPNODE", "ROAD_CODE", "DIR"]
        }
    ],
    "relationships": [
        {
            "type": "HAS_PLAN",
            "from": "event record",
            "to": "event plan",
            "properties": []
        },
        {
            "type": "HAS_COMMAND",
            "from": "event plan",
            "to": "event plan command",
            "properties": []
        },
        {
            "type": "CONTROLS_VMS",
            "from": "event plan command",
            "to": "VMS",
            "properties": []
        }
    ]
}

### Creating Knowledge Graph in Neo4j Database

In [ ]:
def validate_schema(schema):
    required = {"nodes", "relationships"}
    if not required.issubset(schema):
        raise ValueError("Invalid schema format")
    for node in schema["nodes"]:
        if "label" not in node or "properties" not in node:
            raise ValueError("Node missing label/properties")

# Define functions to create nodes from a dataframe
def create_nodes(session, label, df, properties, unique_key):
    """
    For each row in the dataframe, create or merge a node.
    Assumes 'unique_key' is present in each row.
    """
    for _, row in df.iterrows():
        props = {field: row[field] for field in properties if field in row and pd.notnull(row[field])}

        query = f"""
        MERGE (n:{label} {{ {unique_key}: $unique_value }})
        SET n += $props
        """
        session.run(query, unique_value=props.get(unique_key), props=props)


def create_relationships(session, rel_type, from_label, to_label, from_key, to_key, from_df, to_df):
    """
    For each row in the from_df, create a relationship based on the join condition
    that the value in column from_key (e.g., event_record.id) matches the value
    in the to_df's column to_key (e.g., event_plan.event_id).
    """

    target_map = {str(row[to_key]).strip(): str(row[to_key]).strip() 
                  for _, row in to_df.iterrows() if pd.notnull(row[to_key])}
    
    for idx, row in from_df.iterrows():
        source_value = row[from_key]
        if pd.notnull(source_value):
            source_value = str(source_value).strip()
            if source_value in target_map:
                source_value = str(source_value)
                query = f"""
                MATCH (a:{from_label} {{ {from_key}: $source_value }})
                MATCH (b:{to_label} {{ {to_key}: $target_value }})
                MERGE (a)-[r:{rel_type}]->(b)
                """ 

                session.run(query, source_value=source_value, target_value=source_value)
                
            else:
                print(f"Row {idx}: source value '{source_value}' not found in target values.")

def create_relationships_int(session, rel_type, from_label, to_label, from_key, to_key, from_df, to_df):
    """
    For each row in the from_df, create a relationship based on the join condition
    that the value in column from_key (e.g., event_record.id) matches the value
    in the to_df's column to_key (e.g., event_plan.event_id).
    """

    target_map = {str(row[to_key]).strip(): str(row[to_key]).strip() 
                  for _, row in to_df.iterrows() if pd.notnull(row[to_key])}
    
    for idx, row in from_df.iterrows():
        source_value = row[from_key]
        if pd.notnull(source_value):
            source_value = str(source_value).strip()
            if source_value in target_map:
                source_value = str(source_value)
                query = f"""
                MATCH (a:{from_label} {{ {from_key}: {source_value} }})
                MATCH (b:{to_label} {{ {to_key}: {source_value} }})
                MERGE (a)-[r:{rel_type}]->(b)
                """ 
                session.run(query, source_value=source_value, target_value=source_value)
                
            else:
                print(f"Row {idx}: source value '{source_value}' not found in target values.")       

with driver.session() as session:
    create_nodes(session, "event_record", emr, schema["nodes"][0]["properties"], unique_key="id")
    create_nodes(session, "event_plan", emp, schema["nodes"][1]["properties"], unique_key="id")
    create_nodes(session, "event_plan_command", empc, schema["nodes"][2]["properties"], unique_key="id")
    create_nodes(session, "VMS", vms, schema["nodes"][3]["properties"], unique_key="ID")

    create_relationships_int(session, "HAS_PLAN", "event_record", "event_plan", from_key="id", to_key="event_id", from_df=emr, to_df=emp)

    create_relationships_int(session, "HAS_COMMAND", "event_plan", "event_plan_command", from_key="id", to_key="plan_id", from_df=emp, to_df=empc)

    create_relationships(session, "CONTROLS_VMS", "event_plan_command", "VMS", from_key="eqt_site_id", to_key="EQT_EXT_ID", from_df=empc, to_df=vms)
    
print("Knowledge Graph created successfully!")

Knowledge Graph created successfully!


## Road Map

### Data extraction

Get all the unique junction ids

In [14]:
gdf = gpd.read_file("C:/Users/danie/Documents/internship/KG-road/road_map_data/map_nw_road_dubai_new1.shp")

Create nodes for each unique juntion id (REDACTED)

In [ ]:
# jdf = pd.DataFrame({'junction_id': pd.concat([gdf['f_jnctid'], gdf['t_jnctid']]).drop_duplicates().reset_index(drop=True)})
# jdf['junction_id'] = jdf['junction_id'].astype('Int64')


In [ ]:
# # Prepare data for batch creation
# junction_nodes = jdf["junction_id"].astype(str).drop_duplicates().to_list()

# with driver.session() as session:
#     query = """
#     UNWIND $junction_ids AS jid
#     CREATE (j:Junction {junction_id: jid})
#     """
#     session.run(query, junction_ids=junction_nodes)

# print("Junction nodes created successfully!")


Junction nodes created successfully!


Convert int to their respective road type

In [15]:
int_to_rt = {
    0: "Motorway",
    1: "Major Road",
    2: "Other Major Road",
    3: "Secondary Road",
    4: "Local Connecting Road",
    5: "Local Road of High Importance",
    6: "Local Road",
    7: "Local Road of Minor Importance",
    8: "Other Road"
}

gdf["frc"] = gdf["frc"].replace(int_to_rt)

print(gdf["frc"])

0         Local Road of Minor Importance
1                  Local Connecting Road
2                  Local Connecting Road
3         Local Road of Minor Importance
4         Local Road of Minor Importance
                       ...              
236559    Local Road of Minor Importance
236560    Local Road of Minor Importance
236561    Local Road of Minor Importance
236562    Local Road of Minor Importance
236563    Local Road of Minor Importance
Name: frc, Length: 236564, dtype: object


Create nodes for each link id

In [16]:
idf = pd.DataFrame({'link_id': gdf["id"], 'from_junction' : gdf['f_jnctid'], 'to_junction' : gdf['t_jnctid'], 'meters' : gdf['meters'], 'road_type' : gdf['frc']})
idf['link_id'] = idf['link_id'].astype('Int64')
idf['from_junction'] = idf['from_junction'].astype('Int64')
idf['to_junction'] = idf['to_junction'].astype('Int64')


In [17]:
# Prepare data for batch creation
link_nodes = idf[["link_id", "from_junction", "to_junction", "meters", "road_type"]].astype(str).to_dict("records")

with driver.session() as session:
    query = """
    UNWIND $rows AS row
    CREATE (l:Link {
        link_id: row.link_id,
        from_junction: row.from_junction,
        to_junction: row.to_junction,
        meters: row.meters,
        road_type: row.road_type
    })
    """
    session.run(query, rows=link_nodes)

print("Link nodes created successfully!")


Link nodes created successfully!


Relationship will be as followed in graph structure

LINK TO LINK RELATIONSHIPS

In [ ]:
# def create_link_link_relationships(session, df):
#     for _, row in df.iterrows():
#         to_junction = row["to_junction"]

#         query = """
#         MATCH (a:Link {to_junction: $to_junction}), (b:Link {from_junction: $from_junction})
#         CREATE (a)-[:CONNECTED_TO]->(b)
#         """
#         session.run(query, to_junction = str(to_junction), from_junction = str(to_junction))

# with driver.session() as session:
#     create_link_link_relationships(session, idf)

# print("Link-Link relationships created!")

USE THIS


In [42]:
with driver.session() as session:
    query = """
    MATCH (a:Link), (b:Link)
    WHERE a.to_junction = b.from_junction
    MERGE (a)-[r:CONNECTED_TO {junction_id: a.to_junction, weight: toInteger(b.meters)}]->(b)
    """
    session.run(query)

Batch version

In [ ]:
# def create_link_link_relationships_batch(session, df, batch_size=10000):
#     # Prepare all pairs as a list of dicts
#     pairs = [
#         {
#             "to_junction": str(row["to_junction"]),
#             "from_junction": str(row["to_junction"])
#         }
#         for _, row in df.iterrows()
#     ]
#     # Process in batches
#     for i in range(0, len(pairs), batch_size):
#         batch = pairs[i:i+batch_size]
#         query = """
#         UNWIND $pairs AS pair
#         MATCH (a:Link {to_junction: pair.to_junction})
#         MATCH (b:Link {from_junction: pair.from_junction})
#         MERGE (a)-[:CONNECTED_TO]->(b)
#         """
#         session.run(query, pairs=batch)
#         print(f"Processed batch {i//batch_size + 1} / {((len(pairs)-1)//batch_size)+1}")

# with driver.session() as session:
#     create_link_link_relationships_batch(session, idf)

# print("Link-Link relationships created (batched)!")

OLD METHOD

In [ ]:
# def create_link_junction_relationships(session, idf):
#     """
#     For each link in idf, create relationships to its from_junction and to_junction.
#     """
#     for _, row in idf.iterrows():
#         link_id = row["link_id"]
#         from_junction = row["from_junction"]
#         to_junction = row["to_junction"]
        
#         # Relationship from Link to from_junction
#         query_from_to = """
#         MATCH (l:Link {link_id: $link_id})
#         MATCH (f:Junction {junction_id: $from_junction})
#         MATCH (t:Junction {junction_id: $to_junction})
#         MERGE (f)-[:FROM_JUNCTION]->(l)
#         MERGE (l)-[:TO_JUNCTION]->(t)
#         """
#         session.run(query_from_to, link_id=str(link_id), from_junction=str(from_junction), to_junction = str(to_junction))
        

# with driver.session() as session:
#     create_link_junction_relationships(session, idf)

# print("Link-Junction relationships created successfully!")

### Connecting Road Map to Events

Join VMS and event record to their respective links through link_id

In [43]:
def create_vms_link_relationship(session, df):
    for _, row in df.iterrows():
        link_id = row["LINK_ID"]

        query = """
        MATCH (v:VMS {LINK_ID: $link_id})
        MATCH (l:Link {link_id: $target_link})
        CREATE (v)-[:LOCATED_AT {weight: 0}]->(l)
        """
        session.run(query, link_id = int(link_id), target_link = str(link_id))

with driver.session() as session:
    create_vms_link_relationship(session, vms)

print("VMS-Link relationships created!")

VMS-Link relationships created!


In [33]:
def create_event_link_relationship(session, df):
    for _, row in df.iterrows():
        up_link_id = row["up_link_id"]
        dn_link_id = row["dn_link_id"]
        q_link_id = row["q_end_link_id"]

        up_query = """
        MATCH (e:event_record {up_link_id: $link_id})
        MATCH (l:Link {link_id: $target_link})
        CREATE (e)-[:START_AT]->(l)
        """
        session.run(up_query, link_id = int(up_link_id), target_link = str(up_link_id))

        q_query = """
        MATCH (e:event_record {q_end_link_id: $link_id})
        MATCH (l:Link {link_id: $target_link})
        CREATE (e)-[:Q_END_AT]->(l)
        """
        session.run(q_query, link_id = int(q_link_id), target_link = str(q_link_id))

        dn_query = """
        MATCH (e:event_record {dn_link_id: $link_id})
        MATCH (l:Link {link_id: $target_link})
        CREATE (e)-[:END_AT]->(l)
        """
        session.run(dn_query, link_id = int(dn_link_id), target_link = str(dn_link_id))

with driver.session() as session:
    create_event_link_relationship(session, emr)

print("Event-Link Relationships created!")

Event-Link Relationships created!


## MISC CODE


In [ ]:
# def find_vms_for_affected_area(affected_links_result: str) -> str:
#         """
#         Find VMS signs for all affected links from FindAffectedAreaComprehensive output
#         Input: affected_links_result (string) - Output from FindAffectedAreaComprehensive
#         Returns: List of VMS signs covering the affected area
#         """
#         import ast
        
#         try:
#             # Parse the result from FindAffectedAreaComprehensive
#             parsed_result = ast.literal_eval(affected_links_result)
#             if isinstance(parsed_result, list) and len(parsed_result) > 0:
#                 affected_links = parsed_result[0].get('affected_links', [])
#             else:
#                 return "Error: Could not parse affected links result"
            
#             if not affected_links:
#                 return "No affected links found"
            
#             all_vms = []
            
#             with driver.session() as session:
#                 # Query VMS for all affected links at once
#                 query = """
#                 UNWIND $link_ids as link_id
#                 MATCH path = (l:Link {link_id: link_id})-[:CONNECTED_TO*0..2]-(connected:Link)<-[:LOCATED_AT]-(v:VMS)
#                 WITH v, connected, l, path, length(path) as hops, link_id,
#                     CASE 
#                     WHEN length(path) = 0 THEN 0
#                     ELSE reduce(total = 0, node IN nodes(path)[0..-1] | 
#                                 total + coalesce(node.meters, 0))
#                     END as cumulative_distance
                
#                 WITH v, connected, l, cumulative_distance, hops, link_id,
#                     CASE 
#                     WHEN hops = 0 THEN 'same_link'
#                     WHEN hops = 1 THEN 'adjacent'
#                     WHEN hops = 2 THEN 'nearby'
#                     ELSE 'distant'
#                     END as proximity,
#                     relationships(path) as rels
                
#                 WITH v, connected, l, cumulative_distance, hops, proximity, link_id,
#                     CASE 
#                     WHEN hops = 0 THEN 'same_link'
#                     WHEN hops > 0 AND size(rels) > 0 THEN
#                         CASE 
#                         WHEN startNode(rels[0]) = l THEN 'downstream'
#                         WHEN endNode(rels[0]) = l THEN 'upstream'
#                         ELSE 'cross_connection'
#                         END
#                     ELSE 'unknown'
#                     END as direction
                
#                 RETURN DISTINCT v.EQT_NO as vms_id, 
#                     v.ROAD_NAME as road, 
#                     connected.link_id as vms_link,
#                     link_id as affected_link,
#                     cumulative_distance as distance_meters,
#                     hops as links_away,
#                     direction,
#                     proximity
#                 ORDER BY cumulative_distance, hops
#                 LIMIT 50
#                 """
                
#                 result = session.run(query, link_ids=affected_links)
#                 return str([dict(record) for record in result])
                
#         except Exception as e:
#             return f"Error processing affected links: {str(e)}"